# Subtask 1 Accuracy +/-1 Review

Objective:
- Review validation predictions using only the competition-relevant `accuracy_pm1` rule.
- Find examples where the model is exactly right, close-enough right, and clearly wrong.
- Build a stable good-example set for visual inspection and model comparison.


## 0. Setup

This notebook reads saved script artifacts from `results/subtask1/val_preds`. It does not train or run inference.


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'scripts').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from the ai4agri repository root or a child directory')

RESULTS_DIR = REPO_ROOT / 'results' / 'subtask1'
VAL_PREDS_DIR = RESULTS_DIR / 'val_preds'
RUNS_DIR = RESULTS_DIR / 'vision_runs'

# Default to the PM1-trained run; change this to compare other saved runs.
RUN_ID = 'existing_unet_pm1bce_nsum_summary_rand_e10_p1024_v256_m5'
# RUN_ID = 'existing_unet_pm1bce_softnsum_summary_rand_e10_p1024_v256_m5'
# RUN_ID = 'existing_unet_ce_summary_rand_e10_p1024_v256_m5'
# RUN_ID = 'existing_unet_ce_summary_rand_e8_p2048_v512_m5'

VAL_PROBS = VAL_PREDS_DIR / f'{RUN_ID}_val_probs.npz'
METRICS_JSON = RUNS_DIR / RUN_ID / 'metrics.json'

CLASS_NAMES = ['Very Low (0)', 'Low (1)', 'Medium (2)', 'High (3)', 'Very High (4)']
CLASS_COLORS = ['#d73027', '#fc8d59', '#fee08b', '#91cf60', '#1a9850', '#7f7f7f']
TOLERANCE_LABELS = ['Exact', 'Within +/-1', 'Miss >1', 'Ignored']
TOLERANCE_COLORS = ['#1a9850', '#fee08b', '#d73027', '#4d4d4d']

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.size'] = 10

print('Repo root:', REPO_ROOT)
print('Run id:', RUN_ID)
print('Validation payload exists:', VAL_PROBS.exists())
print('Metrics exists:', METRICS_JSON.exists())


## 1. Helpers

`accuracy_pm1` counts exact matches and one-class ordinal misses as correct. Pixels outside classes `0..4` are ignored.


In [ ]:
def load_payload(path: Path) -> dict[str, np.ndarray]:
    if not path.exists():
        raise FileNotFoundError(path)
    payload = np.load(path, allow_pickle=True)
    return {key: payload[key] for key in payload.files}


def display_mask(mask):
    out = np.asarray(mask, dtype=np.int16).copy()
    out[out == 255] = 5
    out[(out < 0) | (out > 5)] = 5
    return out


def valid_mask(y_true, y_pred):
    return (y_true >= 0) & (y_true <= 4) & (y_pred >= 0) & (y_pred <= 4)


def pm1_outcome(y_true, y_pred):
    true = np.asarray(y_true, dtype=np.int16)
    pred = np.asarray(y_pred, dtype=np.int16)
    valid = valid_mask(true, pred)
    diff = np.abs(true - pred)
    out = np.full(true.shape, 3, dtype=np.uint8)
    out[valid & (diff > 1)] = 2
    out[valid & (diff == 1)] = 1
    out[valid & (diff == 0)] = 0
    return out


def patch_stats(patch_ids, y_true, y_pred, probs=None):
    valid = valid_mask(y_true, y_pred)
    diff = np.abs(y_true.astype(np.int16) - y_pred.astype(np.int16))
    valid_counts = valid.sum(axis=(1, 2))
    exact = ((diff == 0) & valid).sum(axis=(1, 2))
    pm1 = ((diff <= 1) & valid).sum(axis=(1, 2))
    miss_gt1 = ((diff > 1) & valid).sum(axis=(1, 2))
    mae_num = np.where(valid, diff, 0).sum(axis=(1, 2))
    rows = {
        'patch_id': patch_ids.astype(str),
        'valid_pixels': valid_counts,
        'exact_rate': np.divide(exact, valid_counts, out=np.full(len(valid_counts), np.nan), where=valid_counts > 0),
        'accuracy_pm1': np.divide(pm1, valid_counts, out=np.full(len(valid_counts), np.nan), where=valid_counts > 0),
        'off_by_one_rate': np.divide(pm1 - exact, valid_counts, out=np.full(len(valid_counts), np.nan), where=valid_counts > 0),
        'miss_gt1_rate': np.divide(miss_gt1, valid_counts, out=np.full(len(valid_counts), np.nan), where=valid_counts > 0),
        'mae': np.divide(mae_num, valid_counts, out=np.full(len(valid_counts), np.nan), where=valid_counts > 0),
    }
    if probs is not None:
        rows['mean_conf'] = probs.max(axis=1).mean(axis=(1, 2))
    return pd.DataFrame(rows)


def add_tolerance_legend(ax):
    handles = [
        mpatches.Patch(facecolor=color, edgecolor='black', linewidth=0.3, label=label)
        for label, color in zip(TOLERANCE_LABELS, TOLERANCE_COLORS)
    ]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.24), ncol=2, frameon=False, fontsize=8)


def plot_patch_triplets(example_df, title, limit=8):
    example_df = example_df.head(limit).copy()
    if example_df.empty:
        print('No examples for:', title)
        return
    class_cmap = plt.matplotlib.colors.ListedColormap(CLASS_COLORS)
    tol_cmap = plt.matplotlib.colors.ListedColormap(TOLERANCE_COLORS)
    rows = len(example_df)
    fig, axes = plt.subplots(rows, 3, figsize=(12, 3.7 * rows))
    axes = np.atleast_2d(axes)
    for row_idx, row in enumerate(example_df.itertuples(index=False)):
        patch_idx = int(row.patch_index)
        axes[row_idx, 0].imshow(display_mask(y_true[patch_idx]), cmap=class_cmap, vmin=0, vmax=5, interpolation='nearest')
        axes[row_idx, 0].set_title(f'{row.patch_id} truth')
        axes[row_idx, 1].imshow(display_mask(y_pred[patch_idx]), cmap=class_cmap, vmin=0, vmax=5, interpolation='nearest')
        axes[row_idx, 1].set_title(f'pred | pm1={row.accuracy_pm1:.3f}')
        axes[row_idx, 2].imshow(pm1_outcome(y_true[patch_idx], y_pred[patch_idx]), cmap=tol_cmap, vmin=0, vmax=3, interpolation='nearest')
        axes[row_idx, 2].set_title(f'exact={row.exact_rate:.3f}, off1={row.off_by_one_rate:.3f}, miss={row.miss_gt1_rate:.3f}')
        add_tolerance_legend(axes[row_idx, 2])
        for ax in axes[row_idx]:
            ax.axis('off')
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    plt.show()


## 2. Load Predictions

The global score printed here should match `metrics.json` for the chosen run, modulo rounding.


In [ ]:
payload = load_payload(VAL_PROBS)
patch_ids = payload['patch_ids'].astype(str)
probs = payload.get('probs')
y_true = payload['y_true'].astype(np.int16)
y_pred = payload['y_pred'].astype(np.int16)

stats = patch_stats(patch_ids, y_true, y_pred, probs=probs)
stats['patch_index'] = np.arange(len(stats))
stats_valid = stats[stats['valid_pixels'] > 0].copy()

valid = valid_mask(y_true, y_pred)
diff = np.abs(y_true.astype(np.int16) - y_pred.astype(np.int16))
global_exact = float(((diff == 0) & valid).sum() / valid.sum())
global_pm1 = float(((diff <= 1) & valid).sum() / valid.sum())
global_miss = 1.0 - global_pm1
global_mae = float(np.nanmean(np.where(valid, diff, np.nan)))

metrics = json.loads(METRICS_JSON.read_text()) if METRICS_JSON.exists() else {}
summary = pd.Series({
    'run_id': RUN_ID,
    'patches': len(stats),
    'patches_with_valid_pixels': len(stats_valid),
    'global_exact': global_exact,
    'global_accuracy_pm1': global_pm1,
    'global_miss_gt1': global_miss,
    'global_mae': global_mae,
    'metrics_json_accuracy_pm1': metrics.get('accuracy_pm1'),
    'metrics_json_best_epoch': metrics.get('best_epoch'),
})
print(summary.to_string())
display(stats_valid.sort_values('accuracy_pm1', ascending=False).head(10))


## 3. Accuracy +/-1 Distribution

Use this to judge whether the run has broad close-enough performance or is carried by a few easy patches.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(['Exact', 'Off by 1', 'Miss >1'], [global_exact, global_pm1 - global_exact, global_miss], color=TOLERANCE_COLORS[:3])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Share of valid pixels')
axes[0].set_title('Global pixel outcomes')
axes[0].grid(axis='y', alpha=0.2)

axes[1].hist(stats_valid['accuracy_pm1'], bins=25, color='#1a9850', alpha=0.8)
axes[1].axvline(stats_valid['accuracy_pm1'].median(), color='black', ls='--', lw=1, label='median')
axes[1].set_title('Patch Accuracy +/-1')
axes[1].set_xlabel('Patch accuracy_pm1')
axes[1].set_ylabel('Patch count')
axes[1].legend(frameon=False)

axes[2].scatter(stats_valid['exact_rate'], stats_valid['accuracy_pm1'], s=14, alpha=0.5, color='#4c72b0', edgecolors='none')
axes[2].plot([0, 1], [0, 1], color='black', lw=1, alpha=0.4)
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].set_title('Exact vs Accuracy +/-1')
axes[2].set_xlabel('Exact rate')
axes[2].set_ylabel('Accuracy +/-1')
axes[2].grid(alpha=0.2)

fig.tight_layout()
plt.show()


## 4. Good Example Set

Good examples are patches where `accuracy_pm1` is high. The second group is especially useful: exact accuracy is not high, but most errors are only off by one, which is exactly what the metric rewards.


In [ ]:
GOOD_PM1_THRESHOLD = 0.95
LOW_EXACT_THRESHOLD = 0.55
MIN_VALID_PIXELS = 1_000

high_pm1 = stats_valid[
    (stats_valid['accuracy_pm1'] >= GOOD_PM1_THRESHOLD)
    & (stats_valid['valid_pixels'] >= MIN_VALID_PIXELS)
].sort_values(['accuracy_pm1', 'miss_gt1_rate', 'valid_pixels'], ascending=[False, True, False])

metric_good = stats_valid[
    (stats_valid['accuracy_pm1'] >= 0.85)
    & (stats_valid['exact_rate'] <= LOW_EXACT_THRESHOLD)
    & (stats_valid['off_by_one_rate'] >= 0.30)
    & (stats_valid['valid_pixels'] >= MIN_VALID_PIXELS)
].sort_values(['accuracy_pm1', 'off_by_one_rate'], ascending=[False, False])

balanced_good = pd.concat([
    high_pm1.head(8),
    metric_good.head(8),
], ignore_index=True).drop_duplicates('patch_id')

print('High pm1 examples:')
display(high_pm1.head(12))
print('Metric-good examples: high pm1, lower exact, many off-by-one pixels:')
display(metric_good.head(12))
print('Curated good example patch ids:')
GOOD_EXAMPLE_PATCH_IDS = balanced_good['patch_id'].head(12).tolist()
print(GOOD_EXAMPLE_PATCH_IDS)


In [ ]:
plot_patch_triplets(high_pm1, f'{RUN_ID}: high Accuracy +/-1 examples', limit=6)
plot_patch_triplets(metric_good, f'{RUN_ID}: metric-good off-by-one examples', limit=6)


## 5. Edge-Class Good Examples

For PM1-optimized decoders, predicting class `1` for true `0` and class `3` for true `4` is still correct. These examples check whether that behavior is actually happening cleanly.


In [ ]:
def edge_pm1_rates_for_patch(i):
    true = y_true[i]
    pred = y_pred[i]
    row = {}
    mask0 = true == 0
    mask4 = true == 4
    row['true0_pixels'] = int(mask0.sum())
    row['true4_pixels'] = int(mask4.sum())
    row['true0_to_1_rate'] = float((pred[mask0] == 1).mean()) if mask0.any() else np.nan
    row['true4_to_3_rate'] = float((pred[mask4] == 3).mean()) if mask4.any() else np.nan
    row['true0_bad_rate'] = float((np.abs(pred[mask0].astype(np.int16) - 0) > 1).mean()) if mask0.any() else np.nan
    row['true4_bad_rate'] = float((np.abs(pred[mask4].astype(np.int16) - 4) > 1).mean()) if mask4.any() else np.nan
    return row

edge_rows = []
for row in stats_valid.itertuples(index=False):
    edge = edge_pm1_rates_for_patch(int(row.patch_index))
    edge_rows.append({**row._asdict(), **edge})
edge_stats = pd.DataFrame(edge_rows)

true0_good = edge_stats[
    (edge_stats['true0_pixels'] >= 500)
    & (edge_stats['true0_bad_rate'] <= 0.05)
].sort_values(['true0_bad_rate', 'true0_to_1_rate'], ascending=[True, False])

true4_good = edge_stats[
    (edge_stats['true4_pixels'] >= 500)
    & (edge_stats['true4_bad_rate'] <= 0.05)
].sort_values(['true4_bad_rate', 'true4_to_3_rate'], ascending=[True, False])

print('Good true-0 edge examples: mostly predicted as 0/1')
display(true0_good[['patch_id', 'accuracy_pm1', 'exact_rate', 'true0_pixels', 'true0_to_1_rate', 'true0_bad_rate']].head(10))
print('Good true-4 edge examples: mostly predicted as 3/4')
display(true4_good[['patch_id', 'accuracy_pm1', 'exact_rate', 'true4_pixels', 'true4_to_3_rate', 'true4_bad_rate']].head(10))


In [ ]:
plot_patch_triplets(true0_good, f'{RUN_ID}: good true-0 +/-1 edge behavior', limit=4)
plot_patch_triplets(true4_good, f'{RUN_ID}: good true-4 +/-1 edge behavior', limit=4)


## 6. Failure Set

These are the patches to inspect before submission. Red pixels are the only pixels that hurt `accuracy_pm1`.


In [ ]:
worst_pm1 = stats_valid.sort_values(['accuracy_pm1', 'miss_gt1_rate', 'valid_pixels'], ascending=[True, False, False])
high_conf_bad = stats_valid.sort_values(['miss_gt1_rate', 'accuracy_pm1'], ascending=[False, True])

print('Worst by accuracy_pm1:')
display(worst_pm1.head(12))
plot_patch_triplets(worst_pm1, f'{RUN_ID}: worst Accuracy +/-1 failures', limit=8)


## 7. Compare Runs By Accuracy +/-1

This table scans all local `val_probs` artifacts and ranks them by the same pixel-level PM1 logic.


In [ ]:
rows = []
for path in sorted(VAL_PREDS_DIR.glob('*_val_probs.npz')):
    rid = path.name.removesuffix('_val_probs.npz')
    try:
        p = load_payload(path)
        yt = p['y_true'].astype(np.int16)
        yp = p['y_pred'].astype(np.int16)
        v = valid_mask(yt, yp)
        d = np.abs(yt - yp)
        rows.append({
            'run_id': rid,
            'patches': yt.shape[0],
            'valid_pixels': int(v.sum()),
            'accuracy_pm1': float(((d <= 1) & v).sum() / v.sum()),
            'exact_rate': float(((d == 0) & v).sum() / v.sum()),
            'miss_gt1_rate': float(((d > 1) & v).sum() / v.sum()),
            'mae': float(np.nanmean(np.where(v, d, np.nan))),
        })
    except Exception as exc:
        rows.append({'run_id': rid, 'error': repr(exc)})

comparison = pd.DataFrame(rows).sort_values('accuracy_pm1', ascending=False, na_position='last')
display(comparison)
